In [1]:
import pandas as pd

PATH_TO_DATA = "./sudar-application-log(windows)-2.csv"
log_df = pd.read_csv(PATH_TO_DATA)

In [2]:
# data cleaning
# check missing value

# print the datatype of eac column
column_summary = pd.DataFrame({
    "missing_value": log_df.isnull().sum(),
    "datatype": log_df.dtypes
})

column_summary

,missing_value,datatype
Level,0,str
Date and Time,6,str
Source,6,str
Event ID,6,float64
Task Category,592,str
Description,6,str


In [3]:
# handle strings
# string_columns = cleaned_log_df.select_dtypes(include=["str", "object"]).columns
string_columns = column_summary[(column_summary["datatype"] == "str") & (column_summary["missing_value"] > 0)].index.tolist()
print(f"The string columns are {string_columns}")

cleaned_log_df = log_df.copy()

for col in string_columns:

    if cleaned_log_df[col].isna().sum() > len(cleaned_log_df) * 0.5:
        cleaned_log_df = cleaned_log_df.drop(columns=[col])
    # logs are collected in order so ffil for data will be correct
    elif "date" in col.lower() or "time" in col.lower():
        cleaned_log_df[col] = pd.to_datetime(cleaned_log_df[col], errors="coerce").ffill()

    else:
        mode = cleaned_log_df[col].mode()
        if not mode.empty:
            cleaned_log_df[col] = cleaned_log_df[col].fillna(mode.iloc[0])

# handling numeric columns

cleaned_log_df["Event ID"] = cleaned_log_df["Event ID"].fillna(cleaned_log_df["Event ID"].median())

cleaned_df_summary = pd.DataFrame({
    "missing_value": cleaned_log_df.isna().sum(),
    "datatype": cleaned_log_df.dtypes
})

print(cleaned_df_summary)

The string columns are ['Date and Time', 'Source', 'Task Category', ' Description']
               missing_value        datatype
Level                      0             str
Date and Time              0  datetime64[us]
Source                     0             str
Event ID                   0         float64
 Description               0             str


In [4]:
unique  = [val.strip() for val in cleaned_log_df['Level'].unique().tolist()]
print(unique)

['Information', 'Warning', 'Error', 'DPTF Build Version:  9.1.10009.1745    DPTF Build Date:  Mar 11 2026 16:58:52    Message:  EPO Modes Status:DC Endurance Gaming(Status: Enabled);DC Better Battery(Status: Disabled);DC Balanced(Status: Enabled);AC Balanced(Status: Enabled);AC Quiet(Status: Disabled);AC Collaboration(Status: Enabled    Policy:  Energy Performance Optimizer Policy [2]']


In [5]:
df_required = cleaned_log_df[cleaned_log_df['Level'].isin(['Error', 'Warning'])]
df_required.reset_index(drop=True)

,Level,Date and Time,Source,Event ID,Description
0,Warning,2026-07-07 21:27:08,VBScriptDeprecationAlert,4096.0,The description for Event ID 4096 from source ...
1,Error,2026-07-07 20:08:19,Microsoft-Windows-RestartManager,10006.0,Application or service 'Microsoft Office SDX H...
2,Error,2026-06-07 16:52:28,VSS,8194.0,Volume Shadow Copy Service error: Unexpected e...
3,Error,2026-06-07 11:58:37,VSS,8194.0,Volume Shadow Copy Service error: Unexpected e...
4,Error,2026-05-07 18:52:04,VSS,8194.0,Volume Shadow Copy Service error: Unexpected e...
5,Error,2026-05-07 18:52:04,SDSSnapshotProcess,277.0,The description for Event ID 277 from source S...
6,Error,2026-03-07 21:48:44,VSS,8194.0,Volume Shadow Copy Service error: Unexpected e...


In [6]:
print(f"The total number of Error Entries is: {len(log_df[log_df['Level'] == 'Error'])} and Warning entries is {len(log_df[log_df['Level'] == 'Warning'])}")

The total number of Error Entries is: 6 and Warning entries is 1


In [18]:
# generate report 
report = df_required.groupby(['Level', 'Source']).size().reset_index(name='Count').sort_values(by='Count', ascending=False)
report["Total Count (by Level)"] = report.groupby('Level')['Count'].transform('sum')
report["Percentage"] = (report["Count"] / report["Total Count (by Level)"]) * 100

report.reset_index(drop=True, inplace=True)
report

,Level,Source,Count,Total Count (by Level),Percentage
0,Error,VSS,4,6,66.666667
1,Error,Microsoft-Windows-RestartManager,1,6,16.666667
2,Error,SDSSnapshotProcess,1,6,16.666667
3,Warning,VBScriptDeprecationAlert,1,1,100.000000


In [19]:
report.to_csv("log_analysis_report.csv", index=False)